# 04 · Validate the build
Automated checks that the pipeline produced complete, physically plausible outputs. The notebook exits with a JSON verdict
(`{"passed": n, "failed": [...]}`), so it can gate a job or CI run.

In [ ]:
%pip install -q "geobrix[light_env6] @ https://github.com/databrickslabs/geobrix/releases/download/v0.5.2/geobrix-0.5.2-py3-none-any.whl"

In [ ]:
%restart_python

In [ ]:
%run ./00_config

In [ ]:
import glob, json
checks = {}

def check(name, ok, detail):
    checks[name] = {"ok": bool(ok), "detail": detail}
    print(("✅" if ok else "❌"), name, "→", detail)

q = lambda sql: spark.sql(sql).collect()

# ── 1. Ingest completeness ────────────────────────────────────────────────
n_files, min_bytes = q(f"SELECT count(*), min(bytes) FROM {FQ}.bronze_file_manifest")[0]
check("manifest_17_files", n_files == 17 and min_bytes > 100_000, f"{n_files} files, smallest {min_bytes:,} bytes")

# ── 2. Raster tile counts ─────────────────────────────────────────────────
n_monthly = q(f"SELECT count(*) FROM {FQ}.silver_climate_monthly_tiles")[0][0]
check("monthly_tiles_1680", n_monthly == 1680, f"{n_monthly} monthly tiles (14 files × 120 months)")
n_dec = q(f"SELECT count(*) FROM {FQ}.silver_climate_decadal_tiles")[0][0]
n_chg = q(f"SELECT count(*) FROM {FQ}.gold_change_tiles")[0][0]
check("decadal_28_change_24", n_dec == 28 and n_chg == 24, f"{n_dec} decadal tiles, {n_chg} change tiles")

# ── 3. COG outputs exist and are valid Cloud-Optimised GeoTIFFs ───────────
from rio_cogeo.cogeo import cog_validate
cogs = sorted(glob.glob(f"{VOL}/outputs/cog/*.tif"))
valid = [cog_validate(p)[0] for p in cogs]
check("cogs_valid", len(cogs) == 25 and all(valid), f"{len(cogs)} COGs, {sum(valid)} pass cog_validate")

# ── 4. H3 land grid ───────────────────────────────────────────────────────
n_land = q(f"SELECT count(*) FROM {FQ}.silver_h3_land")[0][0]
check("land_cells_plausible", 4000 <= n_land <= 6000, f"{n_land:,} H3 res-{H3_RES} land cells (Australia ≈ 7.7M km² / ~1,770 km²)")

# ── 5. Physical plausibility ──────────────────────────────────────────────
base_t = q(f"SELECT avg(tasmax_ann) FROM {FQ}.gold_h3_climate WHERE scenario='historical'")[0][0]
check("baseline_tasmax_plausible", 26 <= base_t <= 32, f"2001-2010 mean daily max over land = {base_t:.2f} °C")

w = {r[0]: r[1] for r in q(f"SELECT scenario, avg(d_tasmax_ann) FROM {FQ}.gold_h3_climate WHERE period='2091-2100' GROUP BY 1")}
check("warming_order_2090s", 0 < w["ssp126"] < w["ssp370"] < w["ssp585"] < 9,
      f"Δ°C 2091-2100: SSP1-2.6 {w['ssp126']:.2f} < SSP3-7.0 {w['ssp370']:.2f} < SSP5-8.5 {w['ssp585']:.2f}")

pr_base = q(f"SELECT avg(pr_ann_mm) FROM {FQ}.gold_h3_climate WHERE scenario='historical'")[0][0]
check("baseline_rain_plausible", 300 <= pr_base <= 900, f"2001-2010 mean annual rainfall over land cells = {pr_base:.0f} mm")

# ── 6. Join integrity: population ─────────────────────────────────────────
pop_total, unmatched = q(f"""SELECT sum(population), count_if(cell_id NOT IN (SELECT cell_id FROM {FQ}.silver_h3_land))
                             FROM {FQ}.silver_sa2_cell""")[0]
check("population_joined", abs(pop_total - 27.6e6) / 27.6e6 < 0.02 and unmatched == 0,
      f"{pop_total:,} people mapped to H3 (ABS ERP 2025 ≈ 27.6M), {unmatched} SA2s unmatched")

exp = {r[0]: r[1] for r in q(f"""SELECT scenario, pop_summer_max_35c FROM {FQ}.gold_exposure_summary
                                 WHERE state='Australia' AND period='2091-2100'""")}
check("exposure_monotonic", exp["ssp126"] < exp["ssp370"] < exp["ssp585"],
      f"People with summer max ≥35 °C by 2091-2100: {exp['ssp126']:,} / {exp['ssp370']:,} / {exp['ssp585']:,}")

# ── 7. Reusable method + app tables ───────────────────────────────────────
n_fn = q(f"SELECT count(*) FROM {FQ}.heat_exposure('ssp370', '2091-2100', 3.0)")[0][0]
check("uc_function_works", n_fn >= 8, f"heat_exposure() returned {n_fn} state rows")
n_states = q(f"SELECT count(*) FROM {FQ}.gold_state_outlines")[0][0]
check("state_outlines", n_states == 8, f"{n_states} state/territory outlines (6 states + NT + ACT)")

In [ ]:
failed = [k for k, v in checks.items() if not v["ok"]]
print(f"\n{len(checks) - len(failed)}/{len(checks)} checks passed")
dbutils.notebook.exit(json.dumps({"passed": len(checks) - len(failed), "total": len(checks), "failed": failed,
                                  "details": {k: v["detail"] for k, v in checks.items()}}))